# Module 2.3: Feature Scaling (Standard, MinMax, Robust) under Sensor Spikes
### 🇵🇱 Odporne Skalowanie Cech w Obecności Szpilek Czujników

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/m02_l03_feature_scaling_spikes.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Evaluate StandardScaler, MinMaxScaler, and RobustScaler on industrial motor telemetry contaminated with 5,000A electrical short-circuit spikes.  
> 🇵🇱 **Streszczenie:** Porównaj zachowanie StandardScaler, MinMaxScaler i RobustScaler na danych silnika zaburzonych impulsami zwarciowymi o wartości 5000A.

- **Estimated Time:** 25 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **The Inrush Current Anomaly**: When heavy industrial induction motors start, they draw $6\times-10\times$ their nominal full-load current for a few hundred milliseconds.
2. **Breakdown of StandardScaler**:
   $$z = \frac{x - \mu}{\sigma}$$
   A single startup surge of $5,000\text{ A}$ inflates $\sigma$ drastically. Normal operational readings ($35\text{ A} \pm 5\text{ A}$) are crushed into an indistinguishable range ($z \in [0.001, 0.003]$).
3. **Breakdown of MinMaxScaler**: The entire operational variance is squished into the bottom $1\%$ of $[0, 1]$.
4. **RobustScaler Immunity**: Uses Median and Interquartile Range (IQR):
   $$x' = \frac{x - \text{median}}{\text{IQR}}$$
   Because Median and IQR ignore the outer $25\%$ tails, operational resolution remains 100% intact.


In [ ]:
# 1. Simulate Motor Current with 5,000A Inrush Surges
np.random.seed(42)
n_pts = 1000

# Normal steady-state operation: Mean = 35.0 A, Std = 3.5 A
normal_amps = np.random.normal(loc=35.0, scale=3.5, size=n_pts)

# Inject startup inrush current spikes (1,500A - 5,000A)
surge_indices = np.random.choice(n_pts, size=10, replace=False)
raw_telemetry = normal_amps.copy()
raw_telemetry[surge_indices] = np.random.uniform(1500.0, 5000.0, size=len(surge_indices))

df_amps = pd.DataFrame({"motor_amps": raw_telemetry})
print(f"Summary Statistics with Startup Surges:\n{df_amps.describe().round(1)}")


In [ ]:
# 2. Compare StandardScaler vs MinMaxScaler vs RobustScaler
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()
robust_scaler = RobustScaler()

df_amps["standard"] = std_scaler.fit_transform(df_amps[["motor_amps"]])
df_amps["minmax"]   = minmax_scaler.fit_transform(df_amps[["motor_amps"]])
df_amps["robust"]   = robust_scaler.fit_transform(df_amps[["motor_amps"]])

# Inspect only the normal operational range (excluding extreme surges)
normal_mask = ~df_amps.index.isin(surge_indices)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].hist(df_amps.loc[normal_mask, "standard"], bins=25, color='#ef4444', edgecolor='black', alpha=0.7)
axes[0].set_title("StandardScaler (Crushed Variance)", fontweight='bold')
axes[0].set_xlabel("Z-Score")

axes[1].hist(df_amps.loc[normal_mask, "minmax"], bins=25, color='#f59e0b', edgecolor='black', alpha=0.7)
axes[1].set_title("MinMaxScaler (Crushed to [0, 0.01])", fontweight='bold')
axes[1].set_xlabel("Normalized Value")

axes[2].hist(df_amps.loc[normal_mask, "robust"], bins=25, color='#10b981', edgecolor='black', alpha=0.7)
axes[2].set_title("RobustScaler (Preserved Variance!)", fontweight='bold')
axes[2].set_xlabel("IQR Units")

plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Inject severe electrical short-circuit spikes into motor current telemetry *(🇵🇱 Zasymuluj skrajne piki zwarciowe w telemetrii prądu pobieranego przez silnik)*
- [ ] **Task 2:** Fit StandardScaler, MinMaxScaler, and RobustScaler pipelines on contaminated data *(🇵🇱 Dopasuj skalery StandardScaler, MinMaxScaler i RobustScaler do zanieczyszczonych danych)*
- [ ] **Task 3:** Inspect variance compression and numerical resolution of normal operating data *(🇵🇱 Zbadaj spłaszczenie wariancji i rozdzielczość poprawnych danych operacyjnych)*


In [ ]:
# Exercise Challenge:
# Prove that RobustScaler maintains an IQR of approximately 1.0 on normal operations
robust_iqr = np.percentile(df_amps.loc[normal_mask, "robust"], 75) - np.percentile(df_amps.loc[normal_mask, "robust"], 25)
print(f"Robust Scaled Normal Operational IQR: {robust_iqr:.3f}")
assert 0.95 <= robust_iqr <= 1.05, "RobustScaler failed to preserve unit IQR!"
print("✓ Robust scaling stability test passed.")
